# Ready to resume — two-hour daytime run

**Continue from 17,371 saved games.** The staged simulator repairs are installed. Your learned model, optimizer, random state and older opponents are preserved.

- **Opponent mix:** 60% current self-play, 30% older models, 10% heuristic.
- **Decks:** the same 176 decks across all 11 classes.
- **Compute:** six learning threads, four game collectors and four evaluation workers, locally.
- **Session limit:** two hours per launch; in-flight work and checkpoint saving may finish afterward.
- **Progress:** the dashboard carries forward lifetime games and strength history. Earlier run output is archived with this preparation.

The repaired engine needs a new baseline evaluation before training resumes. Previous evaluation reports stay in your history. Completed tactical lessons carry forward; periodic refreshers remain enabled.

**Choose Run → Run All Cells when ready.** The first code cell verifies the resume. The second launches the run. Training has not been started during preparation.


In [ ]:
from pathlib import Path
import json, subprocess, sys
from IPython.display import HTML, display
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p/'tools/train_neural.py').exists())
if str(ROOT) not in sys.path: sys.path.insert(0,str(ROOT))
from importlib import reload
import tools.lifetime_dashboard as dashboard_module
Dashboard = reload(dashboard_module).Dashboard
PYTHON = ROOT/'.venv/bin/python'
PREFLIGHT_OK = False
subprocess.run([str(PYTHON),str(ROOT/'tools/activate_learning_speed.py'),'--activate'],cwd=ROOT,check=True)
import os
os.environ.pop('HEARTHSTONE_EVALUATION_QUEUE',None)
campaign_tool = ROOT/'tools/long_training_campaign.py'
# Preparation, integrity checks and configuration validation only; no games or lessons.
preflight = subprocess.run([str(PYTHON),str(campaign_tool)],cwd=ROOT,text=True,stdout=subprocess.PIPE,stderr=subprocess.STDOUT)
print(preflight.stdout.splitlines()[0] if preflight.returncode == 0 else preflight.stdout)
if preflight.returncode == 0: print('Resume checks passed. Training has not started.')
PREFLIGHT_OK = preflight.returncode == 0
if not PREFLIGHT_OK: raise RuntimeError('Campaign checks failed. Training will not start.')


In [ ]:
# This cell starts only when YOU run it. No training was started while preparing this notebook.
if not globals().get('PREFLIGHT_OK',False): raise RuntimeError('Run the successful checks first.')
import time
screen = None
dashboard = None
last_redraw = 0.0
redraw_events = {'started','iteration_saved','tactical_finished','evaluation_started','evaluation_finished','complete','stopped'}
with subprocess.Popen([str(PYTHON),str(campaign_tool),'--train'],cwd=ROOT,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True) as process:
    try:
        for line in process.stdout:
            try: event=json.loads(line)
            except ValueError:
                print(line.rstrip());continue
            if event.get('event')=='campaign_ready':
                print('Continuing saved model at',event['global_game'],'games. Remaining campaign games:',event['remaining_games'])
                if event['complete']:print('Campaign complete. No more training will launch.');continue
                dashboard=Dashboard(event['remaining_games'], root=ROOT, start_global=event['global_game'], session_seconds=event['session_seconds'], opponent_mix=event.get('opponent_mix'));dashboard.checkpoint=event['checkpoint']['path'];screen=display(HTML(dashboard.html()),display_id=True)
                print('Baseline evaluation needed:',event['baseline'])
            elif dashboard is not None:
                dashboard.consume(event)
                now=time.monotonic()
                # Collection events still update internal counters, but redraw only at batch boundaries.
                # Evaluation/lesson heartbeats keep long non-batch phases visibly alive.
                if event.get('event') in redraw_events or (event.get('event') in ('evaluation_game','tactical_progress','training_progress','game_finished') and now-last_redraw >= 15):
                    screen.update(HTML(dashboard.html()));last_redraw=now
                if event.get('event') in ('tactical_finished','iteration_saved','evaluation_finished','complete','stopped'):
                    summary={k:event[k] for k in ('event','iteration','global_game','checkpoint','metrics','reason','report','assessment_before','assessment_after') if k in event}
                    print(json.dumps(summary))
        returncode=process.wait()
    except BaseException:
        process.terminate()
        try:process.wait(timeout=10)
        except subprocess.TimeoutExpired:process.kill();process.wait()
        raise
if returncode:raise RuntimeError('Run stopped. Saved checkpoints remain intact; inspect the message above.')
